<a href="https://colab.research.google.com/github/Prit-P2/Python-/blob/main/PYTHON_min.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install catboost
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import balanced_accuracy_score

### Loading Train data and filturing missing value with 'missing'

In [3]:
#loading train data
train_df = pd.read_csv("train.csv")
categer = (train_df.dtypes == 'object')
cat_feature_list = list(categer[categer].index)
train_df[cat_feature_list] = train_df[cat_feature_list].fillna('missing')

### Loadinf Test Data and filtering missing value with 'missing'

In [4]:

#load test data
test_df = pd.read_csv("test.csv")
categer_t = (test_df.dtypes == 'object')
cat_feature_list_t = list(categer_t[categer_t].index)
test_df[cat_feature_list_t] = test_df[cat_feature_list_t].fillna('missing')

### Creating X test dataset

In [5]:

#test X
d_t = ["id","water_intake","diet_type","calorie_expenditure","gender","heart_rate"]
X_test = test_df.drop(columns=d_t, axis=1)
cat_feature_list_t = [c for c in cat_feature_list_t if c not in d_t]

### Creating X(features) and y(lables)

In [6]:
#creating X y train
drop_cols = ["id", "health_condition","water_intake","diet_type","calorie_expenditure","gender","heart_rate"]
X = train_df.drop(columns=drop_cols, axis=1)
cat_feature_list = [c for c in cat_feature_list if c not in drop_cols]
y = train_df["health_condition"]
le = LabelEncoder()

### Spliting according to train, test and validate split

In [7]:
#train test valadation and Pool datastructure
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

### LableEncoding Lables(y) and creating pool data sttructure
- Here Categorical features is automaticaly handeled by catboost



In [8]:
y_train = le.fit_transform(y_train)
y_valid = le.transform(y_valid)
train_pool = Pool(X_train, y_train, cat_features=cat_feature_list)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_feature_list)
test_pool = Pool(X_test, cat_features=cat_feature_list_t)

### Creating Model
- Creating Catboosts model using CatboostClassifer
 - giving it parameters like iteration, learning rate, depth, loss_function, etc.
- Fiting model to train using model.fit()

In [9]:
#model creation
model = CatBoostClassifier(
    iterations=2968,
    learning_rate= 0.05572936652131584,
    depth=6,
    l2_leaf_reg= 8.96746673481334,

    loss_function="MultiClass",
    auto_class_weights="Balanced",
    eval_metric="MultiClass",
    random_state=42,
    task_type="GPU",
    )
model.fit(
    train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=100,
    verbose=100
)

0:	learn: 1.0416979	test: 1.0417902	best: 1.0417902 (0)	total: 29.2ms	remaining: 1m 26s
100:	learn: 0.1910124	test: 0.1894776	best: 0.1894776 (100)	total: 1.58s	remaining: 44.9s
200:	learn: 0.1777138	test: 0.1779778	best: 0.1779778 (200)	total: 2.96s	remaining: 40.8s
300:	learn: 0.1731999	test: 0.1751972	best: 0.1751972 (300)	total: 4.33s	remaining: 38.4s
400:	learn: 0.1703023	test: 0.1741055	best: 0.1741038 (397)	total: 5.75s	remaining: 36.8s
500:	learn: 0.1681852	test: 0.1736982	best: 0.1736951 (482)	total: 9.26s	remaining: 45.6s
600:	learn: 0.1663078	test: 0.1734393	best: 0.1734321 (590)	total: 10.7s	remaining: 42.1s
700:	learn: 0.1645626	test: 0.1733109	best: 0.1733109 (700)	total: 12.1s	remaining: 39.2s
800:	learn: 0.1628332	test: 0.1732024	best: 0.1731603 (785)	total: 13.5s	remaining: 36.6s
900:	learn: 0.1612211	test: 0.1731906	best: 0.1731314 (858)	total: 14.9s	remaining: 34.3s
bestTest = 0.1731313532
bestIteration = 858
Shrink model to first 859 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=6, eval_metric='MultiClass', iterations=2968, l2_leaf_reg=8.96746673481334, learning_rate=0.05572936652131584, loss_function='MultiClass', random_state=42, task_type='GPU')

### Predicting the model output

In [10]:
#predict
val_predict = model.predict(valid_pool)
val_rmse = balanced_accuracy_score(y_valid, val_predict)
print(f"\nValidation balance: {val_rmse:.4f}")


Validation balance: 0.9503


### Predicting on Test dataset

In [11]:
# create submission file using sample_submission.csv format
submission = pd.DataFrame({
    "id": test_df["id"],
    "health_condition": le.inverse_transform(model.predict(test_pool).astype(int))
})
submission.to_csv("submission.csv", index=False)
print("Submission file saved to submission.csv")

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Submission file saved to submission.csv
